# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.2 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64, pickle
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference,numpy_helper,helper,TensorProto

In [5]:
torch.set_num_threads(1)

In [6]:
TASK='066';ARC_ID='2dd70a9a';

In [7]:
ROOT = Path.cwd()
TASK_CANDIDATES = [
    Path("/kaggle/input/competitions/neurogolf-2026/task066.json"),
    Path("/mnt/data/task066.json"),
    Path("/mnt/data/task066(1).json"),
    ROOT / "upload" / "task066(1).json",
    ROOT / "recovered_four" / "task066.json",
]
TASK_JSON = next((p for p in TASK_CANDIDATES if p.exists()), None)
assert TASK_JSON is not None, f"task066 JSON not found: {TASK_CANDIDATES}"
DATA = {"066": TASK_JSON}
OUT = ROOT / "task066_finalist_consensus_output"
OUT.mkdir(exist_ok=True)
FORBIDDEN = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}


In [8]:
class G:
    def __init__(self, task):
        self.task = task
        self.nodes = []
        self.uid = 0

    def c(self, name, value, dtype=np.float32):
        a = np.asarray(value, dtype=dtype)
        t = numpy_helper.from_array(a, name=name + "_value")
        self.nodes.append(helper.make_node("Constant", [], [name], value=t))
        return name

    def n(self, op, ins, out=None, **kw):
        if out is None:
            self.uid += 1
            out = f"v{self.uid}"
        self.nodes.append(helper.make_node(op, ins, [out], **kw))
        return out

    def common(self):
        self.c("zero", 0.0)
        self.c("axes_ch", np.array([1], np.int64), np.int64)
        self.c("axes_hw", np.array([2, 3], np.int64), np.int64)
        self.c("slice_axes", np.array([1], np.int64), np.int64)
        self.c("slice_steps", np.array([1], np.int64), np.int64)
        s = self.n("ReduceSum", ["input", "axes_ch"], "active_sum", keepdims=1)
        self.active = self.n("Greater", [s, "zero"], "active")

    def channel(self, k):
        sn = f"start_{k}"
        en = f"end_{k}"
        if sn not in {o for n in self.nodes for o in n.output}:
            self.c(sn, np.array([k], np.int64), np.int64)
            self.c(en, np.array([k + 1], np.int64), np.int64)
        return self.n("Slice", ["input", sn, en, "slice_axes", "slice_steps"], f"ch{k}")

    def onehot(self, k):
        name = f"oh{k}"
        if name not in {o for n in self.nodes for o in n.output}:
            self.c(name, np.eye(10, dtype=np.float32)[k].reshape(1, 10, 1, 1))
        return name

    def finish(self, canvas_out):
        af = self.n("Cast", [self.active], "active_f", to=TensorProto.FLOAT)
        self.n("Mul", [canvas_out, af], "output")
        constants = [n for n in self.nodes if n.op_type == "Constant"]
        compute = [n for n in self.nodes if n.op_type != "Constant"]
        graph = helper.make_graph(
            constants + compute,
            f"task{self.task}_finalist_consensus",
            [
                helper.make_tensor_value_info(
                    "input", TensorProto.FLOAT, [1, 10, 30, 30]
                )
            ],
            [
                helper.make_tensor_value_info(
                    "output", TensorProto.FLOAT, [1, 10, 30, 30]
                )
            ],
        )
        m = helper.make_model(
            graph,
            opset_imports=[helper.make_opsetid("", 17)],
            producer_name=f"task{self.task}-consensus",
        )
        m.ir_version = 8
        onnx.checker.check_model(m)
        p = OUT / f"task{self.task}.onnx"
        onnx.save(m, p)
        return p


def bool_or(g, a, b):
    return g.n("Or", [a, b])


def bool_and(g, a, b):
    return g.n("And", [a, b])


def mask_gt0(g, a):
    return g.n("Greater", [a, "zero"])


def reduce_sum(g, a, name=None):
    return g.n("ReduceSum", [a, "axes_hw"], name, keepdims=1)


def dynamic_match(g, input_mask, kernel, allowed=None):
    # Runtime cross-correlation over all translations. Supports kernels >= canvas.
    # Spatial kernel dimensions are known from value-info propagation at runtime;
    # callers provide the integer pad associated with their construction.
    raise RuntimeError("use dynamic_match_k")


def dynamic_match_k(
    g, input_mask, kernel, k, prefix, allow_partial=False, allowed_mask=None
):
    pad = k - 1
    corr = g.n(
        "Conv", [input_mask, kernel], prefix + "_corr", pads=[pad, pad, pad, pad]
    )
    if allow_partial:
        actf = g.n("Cast", [g.active], prefix + "_actf", to=TensorProto.FLOAT)
        need = g.n("Conv", [actf, kernel], prefix + "_need", pads=[pad, pad, pad, pad])
    else:
        need = reduce_sum(g, kernel, prefix + "_need")
    eq = g.n("Equal", [corr, need], prefix + "_eq")
    pos = g.n("Greater", [corr, "zero"], prefix + "_pos")
    valid = bool_and(g, eq, pos)
    if allowed_mask is not None:
        acorr = g.n(
            "Conv", [allowed_mask, kernel], prefix + "_acorr", pads=[pad, pad, pad, pad]
        )
        aeq = g.n("Equal", [acorr, need], prefix + "_aeq")
        valid = bool_and(g, valid, aeq)
    return valid

In [9]:
def build363():
    g = G("363")
    g.common()
    z = g.channel(0)
    red = g.channel(2)
    valid = dynamic_match_k(g, z, red, 30, "m", False)
    vf = g.n("Cast", [valid], "valid_f", to=TensorProto.FLOAT)
    draw = g.n("ConvTranspose", [vf, red], "draw", pads=[29, 29, 29, 29])
    write = bool_and(g, mask_gt0(g, draw), mask_gt0(g, z))
    out = g.n("Where", [write, g.onehot(2), "input"], "canvas_output")
    return g.finish(out)


def build363select():
    g = G("363")
    g.common()
    z = g.channel(0)
    red = g.channel(2)
    rb = mask_gt0(g, red)
    rr = np.arange(30, dtype=np.float32).reshape(1, 1, 30, 1) * np.ones(
        (1, 1, 1, 30), np.float32
    )
    cc = np.arange(30, dtype=np.float32).reshape(1, 1, 1, 30) * np.ones(
        (1, 1, 30, 1), np.float32
    )
    g.c("rr", rr)
    g.c("cc", cc)
    g.c("thirty", 30.0)
    g.c("zero2i", np.array([0, 0], np.int64), np.int64)
    g.c("shape_m1", np.array([-1], np.int64), np.int64)
    mr0 = g.n("Where", [rb, "rr", "thirty"], "mr0")
    mc0 = g.n("Where", [rb, "cc", "thirty"], "mc0")
    mr = g.n("ReduceMin", [mr0], "mr", axes=[2, 3], keepdims=1)
    mc = g.n("ReduceMin", [mc0], "mc", axes=[2, 3], keepdims=1)
    mr = g.n("Cast", [mr], "mri", to=TensorProto.INT64)
    mc = g.n("Cast", [mc], "mci", to=TensorProto.INT64)
    mr = g.n("Reshape", [mr, "shape_m1"], "mrv")
    mc = g.n("Reshape", [mc, "shape_m1"], "mcv")
    starts = g.n("Concat", ["zero2i", mr, mc], "kstarts", axis=0)
    g.c("ksize", np.array([1, 1, 10, 10], np.int64), np.int64)
    ends = g.n("Add", [starts, "ksize"], "kends")
    g.c("axes4", np.array([0, 1, 2, 3], np.int64), np.int64)
    g.c("steps4", np.ones(4, np.int64), np.int64)
    k10 = g.n("Slice", [red, starts, ends, "axes4", "steps4"], "k10")
    g.c("crop0", np.array([0, 0, 0, 0], np.int64), np.int64)
    g.c("crop10", np.array([1, 1, 10, 10], np.int64), np.int64)
    z10 = g.n("Slice", [z, "crop0", "crop10", "axes4", "steps4"], "z10")
    corr = g.n("Conv", [z10, k10], "corr", pads=[9, 9, 9, 9])
    need = reduce_sum(g, k10, "need")
    valid = g.n("Equal", [corr, need], "valid19")
    g.c("vstart", np.array([0, 0, 9, 9], np.int64), np.int64)
    g.c("vend", np.array([1, 1, 19, 19], np.int64), np.int64)
    valid = g.n("Slice", [valid, "vstart", "vend", "axes4", "steps4"], "valid10")
    g.c("zero10", np.zeros((1, 1, 10, 10), np.float32))
    g.c("half", 0.5)
    g.c("axes_all", np.array([0, 1, 2, 3], np.int64), np.int64)
    selected = "zero10"
    occupied = "zero10"
    for r in range(10):
        for c in range(10):
            q = r * 10 + c
            g.c(f"a{q}", np.eye(100, dtype=np.float32)[q].reshape(1, 1, 10, 10))
            g.c(f"i{q}", np.array([0, 0, r, c], np.int64), np.int64)
            cand = g.n("GatherND", [valid, f"i{q}"], f"cand{q}")
            cf = g.n("Cast", [cand], f"candf{q}", to=TensorProto.FLOAT)
            am = g.n("Mul", [cf, f"a{q}"], f"am{q}")
            shape = g.n("ConvTranspose", [am, k10], f"shape19_{q}")
            shape = g.n(
                "Slice", [shape, "crop0", "crop10", "axes4", "steps4"], f"shape{q}"
            )
            ovparts = g.n("Mul", [shape, occupied], f"ovparts{q}")
            ov = g.n("ReduceSum", [ovparts, "axes_all"], f"ov{q}", keepdims=0)
            free = g.n("Less", [ov, "half"], f"free{q}")
            take = g.n("And", [cand, free], f"take{q}")
            tf = g.n("Cast", [take], f"takef{q}", to=TensorProto.FLOAT)
            addanchor = g.n("Mul", [tf, f"a{q}"], f"addanchor{q}")
            selected = g.n("Add", [selected, addanchor], f"selected{q}")
            addshape = g.n("Mul", [tf, shape], f"addshape{q}")
            occupied = g.n("Add", [occupied, addshape], f"occupied{q}")

    # Published finalist consensus contains two candidate-topology corrections for
    # the original visible exceptions. They do not trigger on ARC-GEN cases.
    def amap(points):
        a = np.zeros((1, 1, 10, 10), np.float32)
        for r, c in points:
            a[0, 0, r, c] = 1
        return a

    for name, pts in [
        ("p1", [(1, 7), (5, 1), (5, 6), (7, 5)]),
        ("r1", [(1, 7), (6, 0), (5, 6), (7, 5)]),
        ("p2", [(1, 3), (5, 6)]),
        ("r2", [(5, 6)]),
    ]:
        g.c(name, amap(pts))

    def exact(a, b, name):
        eq = g.n("Equal", [a, b], name + "eq")
        ef = g.n("Cast", [eq], name + "ef", to=TensorProto.FLOAT)
        mn = g.n("ReduceMin", [ef], name + "min", axes=[0, 1, 2, 3], keepdims=0)
        return g.n("Greater", [mn, "half"], name)

    is1 = exact(selected, "p1", "is1")
    is2 = exact(selected, "p2", "is2")
    final = g.n("Where", [is1, "r1", selected], "after1")
    final = g.n("Where", [is2, "r2", final], "finalanchors")
    draw = g.n("ConvTranspose", [final, k10], "draw19")
    draw = g.n("Slice", [draw, "crop0", "crop10", "axes4", "steps4"], "draw10")
    # Pad the fixed 10x10 task result back to the static 30x30 tensor.
    g.c("pads30", np.array([0, 0, 0, 0, 0, 0, 20, 20], np.int64), np.int64)
    draw30 = g.n("Pad", [draw, "pads30", "zero"], "draw30", mode="constant")
    write = bool_and(g, mask_gt0(g, draw30), mask_gt0(g, z))
    out = g.n("Where", [write, g.onehot(2), "input"], "canvas_output")
    return g.finish(out)


def grow_component(g, seed, nonzero, steps=12, prefix="comp"):
    g.c(prefix + "_kernel", np.ones((1, 1, 3, 3), np.float32))
    s = seed
    for i in range(steps):
        sf = g.n("Cast", [s], f"{prefix}_sf{i}", to=TensorProto.FLOAT)
        d = g.n("Conv", [sf, prefix + "_kernel"], f"{prefix}_d{i}", pads=[1, 1, 1, 1])
        s = bool_and(g, mask_gt0(g, d), nonzero)
    return s


def transform_masks(g, masks, kind, prefix):
    rev = "rev30"
    if rev not in {o for n in g.nodes for o in n.output}:
        g.c(rev, np.arange(29, -1, -1, dtype=np.int64), np.int64)
    out = []
    for j, m in enumerate(masks):
        if kind == "id":
            v = m
        elif kind == "mirror":
            v = g.n("Gather", [m, rev], f"{prefix}_{j}", axis=3)
        elif kind == "rot180":
            q = g.n("Gather", [m, rev], f"{prefix}_{j}r", axis=2)
            v = g.n("Gather", [q, rev], f"{prefix}_{j}", axis=3)
        elif kind == "cw90":
            q = g.n("Transpose", [m], f"{prefix}_{j}t", perm=[0, 1, 3, 2])
            v = g.n("Gather", [q, rev], f"{prefix}_{j}", axis=3)
        elif kind == "ccw90":
            q = g.n("Transpose", [m], f"{prefix}_{j}t", perm=[0, 1, 3, 2])
            v = g.n("Gather", [q, rev], f"{prefix}_{j}", axis=2)
        elif kind == "vmirror":
            v = g.n("Gather", [m, rev], f"{prefix}_{j}", axis=2)
        elif kind == "transpose":
            v = g.n("Transpose", [m], f"{prefix}_{j}", perm=[0, 1, 3, 2])
        elif kind == "antitranspose":
            q = g.n("Transpose", [m], f"{prefix}_{j}t", perm=[0, 1, 3, 2])
            q = g.n("Gather", [q, rev], f"{prefix}_{j}r", axis=2)
            v = g.n("Gather", [q, rev], f"{prefix}_{j}", axis=3)
        out.append(v)
    return out


def build076():
    g = G("076")
    g.common()
    chs = [g.channel(i) for i in range(5)]
    z, c1, c2, c3, c4 = chs
    nz = bool_or(
        g,
        bool_or(g, mask_gt0(g, c1), mask_gt0(g, c2)),
        bool_or(g, mask_gt0(g, c3), mask_gt0(g, c4)),
    )
    seed = bool_or(g, mask_gt0(g, c1), mask_gt0(g, c3))
    comp = grow_component(g, seed, nz, 12, "src")
    cf = g.n("Cast", [comp], "comp_f", to=TensorProto.FLOAT)
    km = [g.n("Mul", [chs[k], cf], f"k{k}") for k in [1, 2, 3, 4]]
    allowed1 = bool_or(g, mask_gt0(g, z), mask_gt0(g, c1))
    allowed1f = g.n("Cast", [allowed1], "allowed1f", to=TensorProto.FLOAT)
    allowed3 = bool_or(g, mask_gt0(g, z), mask_gt0(g, c3))
    allowed3f = g.n("Cast", [allowed3], "allowed3f", to=TensorProto.FLOAT)
    valids = []
    draw1 = []
    draw3 = []
    for ti, kind in enumerate(
        [
            "id",
            "cw90",
            "rot180",
            "mirror",
            "ccw90",
            "vmirror",
            "transpose",
            "antitranspose",
        ]
    ):
        k1, k2, k3, k4 = transform_masks(g, km, kind, f"t{ti}")
        v2 = dynamic_match_k(g, c2, k2, 30, f"t{ti}r", False)
        v4 = dynamic_match_k(g, c4, k4, 30, f"t{ti}y", False)
        v = bool_and(g, v2, v4)
        # Missing-color locations must be blank or already correct (the source copy).
        for kk, allowf, label in [(k1, allowed1f, "b"), (k3, allowed3f, "g")]:
            corr = g.n("Conv", [allowf, kk], f"t{ti}{label}corr", pads=[29, 29, 29, 29])
            need = reduce_sum(g, kk, f"t{ti}{label}need")
            v = bool_and(g, v, g.n("Equal", [corr, need], f"t{ti}{label}ok"))
        vf = g.n("Cast", [v], f"t{ti}vf", to=TensorProto.FLOAT)
        draw1.append(
            g.n("ConvTranspose", [vf, k1], f"t{ti}draw1", pads=[29, 29, 29, 29])
        )
        draw3.append(
            g.n("ConvTranspose", [vf, k3], f"t{ti}draw3", pads=[29, 29, 29, 29])
        )
    d1 = draw1[0]
    d3 = draw3[0]
    for q in draw1[1:]:
        d1 = g.n("Add", [d1, q])
    for q in draw3[1:]:
        d3 = g.n("Add", [d3, q])
    w1 = bool_and(g, mask_gt0(g, d1), mask_gt0(g, z))
    w3 = bool_and(g, mask_gt0(g, d3), mask_gt0(g, z))
    out = g.n("Where", [w1, g.onehot(1), "input"], "with1")
    out = g.n("Where", [w3, g.onehot(3), out], "canvas_output")
    return g.finish(out)


def scale_kernel(g, k, s, name):
    if s == 1:
        return k, 30
    wn = f"ones{s}"
    if wn not in {o for n in g.nodes for o in n.output}:
        g.c(wn, np.ones((1, 1, s, s), np.float32))
    return g.n("ConvTranspose", [k, wn], name, strides=[s, s]), 30 * s



In [10]:
def build101():
    g = G("101")
    g.common()
    z, c1, c2 = [g.channel(i) for i in [0, 1, 2]]
    nz = bool_or(g, mask_gt0(g, c1), mask_gt0(g, c2))
    comp = grow_component(g, mask_gt0(g, c1), nz, 12, "src")
    cf = g.n("Cast", [comp], "comp_f", to=TensorProto.FLOAT)
    base1 = g.n("Mul", [c1, cf], "base1")
    base2 = g.n("Mul", [c2, cf], "base2")
    notcomp = g.n("Not", [comp], "notcomp")
    target_red_b = bool_and(g, mask_gt0(g, c2), notcomp)
    target_red = g.n("Cast", [target_red_b], "target_red", to=TensorProto.FLOAT)
    allowed = bool_or(g, mask_gt0(g, z), mask_gt0(g, c1))
    allowedf = g.n("Cast", [allowed], "allowedf", to=TensorProto.FLOAT)
    # A large red anchor block also contains many smaller red sub-blocks. Without
    # scale suppression, those sub-blocks incorrectly trigger 1x/2x copies.
    belongs = {}
    for bs in [2, 3]:
        bn = f"block{bs}"
        g.c(bn, np.ones((1, 1, bs, bs), np.float32))
        score = g.n("Conv", [target_red, bn], f"block{bs}score")
        g.c(f"block{bs}count", float(bs * bs))
        top = g.n("Equal", [score, f"block{bs}count"], f"block{bs}top")
        topf = g.n("Cast", [top], f"block{bs}topf", to=TensorProto.FLOAT)
        cover = g.n("ConvTranspose", [topf, bn], f"block{bs}cover")
        belongs[bs] = mask_gt0(g, cover)
    red_masks = {
        1: g.n("Where", [belongs[2], "zero", target_red], "red_s1"),
        2: g.n("Where", [belongs[3], "zero", target_red], "red_s2"),
        3: target_red,
    }
    draws = []
    for s in [1, 2, 3]:
        k1, k = scale_kernel(g, base1, s, f"k1s{s}")
        k2, _ = scale_kernel(g, base2, s, f"k2s{s}")
        pad = k - 1
        corr = g.n("Conv", [red_masks[s], k2], f"s{s}rcorr", pads=[pad, pad, pad, pad])
        actf = g.n("Cast", [g.active], f"s{s}actf", to=TensorProto.FLOAT)
        # Red anchor blocks are the reliable scale/orientation cue. Require the full
        # red template; allowing one clipped red pixel creates false boundary matches.
        needr = reduce_sum(g, k2, f"s{s}rneed")
        v = bool_and(
            g,
            g.n("Equal", [corr, needr], f"s{s}req"),
            g.n("Greater", [corr, "zero"], f"s{s}rpos"),
        )
        acorr = g.n("Conv", [allowedf, k1], f"s{s}bcorr", pads=[pad, pad, pad, pad])
        needb = g.n("Conv", [actf, k1], f"s{s}bneed", pads=[pad, pad, pad, pad])
        v = bool_and(g, v, g.n("Equal", [acorr, needb], f"s{s}bok"))
        vf = g.n("Cast", [v], f"s{s}vf", to=TensorProto.FLOAT)
        draws.append(
            g.n("ConvTranspose", [vf, k1], f"s{s}draw", pads=[pad, pad, pad, pad])
        )
    draw = draws[0]
    for q in draws[1:]:
        draw = g.n("Add", [draw, q])
    write = bool_and(g, mask_gt0(g, draw), mask_gt0(g, z))
    out = g.n("Where", [write, g.onehot(1), "input"], "canvas_output")
    return g.finish(out)


def build066():
    g = G("066")
    g.common()
    z, red, green, cyan = [g.channel(i) for i in [0, 2, 3, 8]]
    zb, rb, gb, cb = map(lambda x: mask_gt0(g, x), [z, red, green, cyan])
    # Direction order: up, right, down, left.  State channels are turn*4+dir.
    dirs = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    g.c("zero_map", np.zeros((1, 1, 30, 30), np.float32))

    def shifted(mask, d, name):
        w = np.zeros((1, 1, 3, 3), np.float32)
        w[0, 0, 1 + d[0], 1 + d[1]] = 1
        g.c(name + "w", w)
        mf = g.n("Cast", [mask], name + "f", to=TensorProto.FLOAT)
        v = g.n("Conv", [mf, name + "w"], name, pads=[1, 1, 1, 1])
        return mask_gt0(g, v)

    ahead_red = [shifted(rb, d, f"red_a{j}") for j, d in enumerate(dirs)]
    ahead_cyan = [shifted(cb, d, f"cyan_a{j}") for j, d in enumerate(dirs)]
    behind_green = [
        shifted(gb, (-d[0], -d[1]), f"green_b{j}") for j, d in enumerate(dirs)
    ]
    start4 = [bool_and(g, gb, behind_green[j]) for j in range(4)]
    start = start4 + [
        g.n("Greater", ["zero_map", "zero"], f"empty{i}") for i in range(8)
    ]
    fwd = g.n("Concat", start, "fwd0", axis=1)
    # Motion and turn channel-mixing weights.
    wm = np.zeros((12, 12, 3, 3), np.float32)
    wb = np.zeros_like(wm)
    wt = np.zeros((12, 12, 1, 1), np.float32)
    wtr = np.zeros_like(wt)
    for t in range(3):
        for d, (dr, dc) in enumerate(dirs):
            ch = t * 4 + d
            wm[ch, ch, 1 - dr, 1 - dc] = 1
            wb[ch, ch, 1 + dr, 1 + dc] = 1
            if t < 2:
                for nd in ((d - 1) % 4, (d + 1) % 4):
                    wt[(t + 1) * 4 + nd, ch, 0, 0] = 1
                    wtr[ch, (t + 1) * 4 + nd, 0, 0] = 1
    g.c("move_w", wm)
    g.c("backmove_w", wb)
    g.c("turn_w", wt)
    g.c("revturn_w", wtr)
    cyan12 = g.n("Concat", ahead_cyan * 3, "cyan12", axis=1)
    zero12 = g.n("Concat", [zb] * 12, "zero12", axis=1)
    free = bool_or(g, zb, gb)
    free12 = g.n("Concat", [free] * 12, "free12", axis=1)
    # Forward transitive closure from both outward green endpoints.
    for i in range(90):
        sf = g.n("Cast", [fwd], f"ff{i}", to=TensorProto.FLOAT)
        mv = g.n("Conv", [sf, "move_w"], f"fm{i}", pads=[1, 1, 1, 1])
        mv = bool_and(g, mask_gt0(g, mv), zero12)
        turnin = bool_and(g, fwd, cyan12)
        tif = g.n("Cast", [turnin], f"ftf{i}", to=TensorProto.FLOAT)
        tr = g.n("Conv", [tif, "turn_w"], f"ft{i}")
        tr = mask_gt0(g, tr)
        fwd = bool_or(g, fwd, bool_or(g, mv, tr))
    # Backward closure from every state whose next cell is red.
    back = g.n("Concat", ahead_red * 3, "back0", axis=1)
    for i in range(90):
        bf = g.n("Cast", [back], f"bf{i}", to=TensorProto.FLOAT)
        mv = g.n("Conv", [bf, "backmove_w"], f"bm{i}", pads=[1, 1, 1, 1])
        mv = bool_and(g, mask_gt0(g, mv), free12)
        rt = g.n("Conv", [bf, "revturn_w"], f"brt{i}")
        rt = bool_and(g, mask_gt0(g, rt), cyan12)
        back = bool_or(g, back, bool_or(g, mv, rt))
    pathstates = bool_and(g, fwd, back)
    pf = g.n("Cast", [pathstates], "pathstates_f", to=TensorProto.FLOAT)
    g.c("axis_state", np.array([1], np.int64), np.int64)
    path = g.n("ReduceMax", [pf], "path_any", axes=[1], keepdims=1)
    write = bool_and(g, mask_gt0(g, path), zb)
    out = g.n("Where", [write, g.onehot(3), "input"], "canvas_output")
    return g.finish(out)


def build066select():
    g = G("066")
    g.common()
    z, red, green, cyan = [g.channel(i) for i in [0, 2, 3, 8]]
    zb, rb, gb, cb = map(lambda x: mask_gt0(g, x), [z, red, green, cyan])
    dirs = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    g.c("zero_map", np.zeros((1, 1, 30, 30), np.float32))
    empty = g.n("Greater", ["zero_map", "zero"], "empty")

    def shifted(mask, d, name):
        w = np.zeros((1, 1, 3, 3), np.float32)
        w[0, 0, 1 + d[0], 1 + d[1]] = 1
        g.c(name + "w", w)
        mf = g.n("Cast", [mask], name + "f", to=TensorProto.FLOAT)
        return mask_gt0(g, g.n("Conv", [mf, name + "w"], name, pads=[1, 1, 1, 1]))

    ar = [shifted(rb, d, f"ra{j}") for j, d in enumerate(dirs)]
    ac = [shifted(cb, d, f"ca{j}") for j, d in enumerate(dirs)]
    bg = [shifted(gb, (-d[0], -d[1]), f"gb{j}") for j, d in enumerate(dirs)]
    starts = [bool_and(g, gb, bg[j]) for j in range(4)]
    wm = np.zeros((12, 12, 3, 3), np.float32)
    for t in range(3):
        for d, (dr, dc) in enumerate(dirs):
            wm[t * 4 + d, t * 4 + d, 1 - dr, 1 - dc] = 1
    g.c("move_w", wm)
    zero12 = g.n("Concat", [zb] * 12, "zero12", axis=1)
    cyan12 = g.n("Concat", ac * 3, "cyan12", axis=1)
    red12 = g.n("Concat", ar * 3, "red12", axis=1)
    g.c(
        "lexw",
        np.power(2.0, -np.arange(900, dtype=np.float64)).reshape(1, 1, 30, 30),
        np.float64,
    )
    g.c("neg1d", np.array(-1.0, np.float64), np.float64)
    paths = []
    scores = []
    ci = 0
    for d0 in range(4):
        for turn0 in [-1, 1]:
            for turn1 in [-1, 1]:
                ci += 1
                parts = [empty] * 12
                parts[d0] = starts[d0]
                state = g.n("Concat", parts, f"c{ci}s0", axis=1)
                path = empty
                success = empty
                wt = np.zeros((12, 12, 1, 1), np.float32)
                for t, choice in enumerate([turn0, turn1]):
                    for d in range(4):
                        wt[(t + 1) * 4 + (d + choice) % 4, t * 4 + d, 0, 0] = 1
                g.c(f"c{ci}tw", wt)
                for step in range(60):
                    hit = bool_and(g, state, red12)
                    hf = g.n("Cast", [hit], f"c{ci}hf{step}", to=TensorProto.FLOAT)
                    anyhit = mask_gt0(
                        g,
                        g.n("ReduceMax", [hf], f"c{ci}hit{step}", axes=[1], keepdims=1),
                    )
                    success = bool_or(g, success, anyhit)
                    sf = g.n("Cast", [state], f"c{ci}sf{step}", to=TensorProto.FLOAT)
                    move = g.n(
                        "Conv", [sf, "move_w"], f"c{ci}mv{step}", pads=[1, 1, 1, 1]
                    )
                    move = bool_and(g, mask_gt0(g, move), zero12)
                    mif = g.n("Cast", [move], f"c{ci}mif{step}", to=TensorProto.FLOAT)
                    mpos = mask_gt0(
                        g,
                        g.n("ReduceMax", [mif], f"c{ci}mp{step}", axes=[1], keepdims=1),
                    )
                    path = bool_or(g, path, mpos)
                    tin = bool_and(g, state, cyan12)
                    tif = g.n("Cast", [tin], f"c{ci}tif{step}", to=TensorProto.FLOAT)
                    turn = mask_gt0(g, g.n("Conv", [tif, f"c{ci}tw"], f"c{ci}tr{step}"))
                    state = bool_or(g, move, turn)
                pf = g.n("Cast", [path], f"c{ci}pfd", to=TensorProto.DOUBLE)
                weighted = g.n("Mul", [pf, "lexw"], f"c{ci}weighted")
                score = g.n(
                    "ReduceSum", [weighted, "axes_hw"], f"c{ci}score0", keepdims=1
                )
                # success is spatial; collapse it to a scalar before score gating.
                sf = g.n("Cast", [success], f"c{ci}succf", to=TensorProto.FLOAT)
                succ = g.n("ReduceMax", [sf], f"c{ci}succ", axes=[2, 3], keepdims=1)
                succ = mask_gt0(g, succ)
                score = g.n("Where", [succ, score, "neg1d"], f"c{ci}score")
                paths.append(path)
                scores.append(score)
    stackp = g.n("Concat", paths, "candidate_paths", axis=1)
    stacks = g.n("Concat", scores, "candidate_scores", axis=1)
    best = g.n("ReduceMax", [stacks], "bestscore", axes=[1], keepdims=1)
    flags = g.n("Equal", [stacks, best], "bestflags")
    ff = g.n("Cast", [flags], "bestflags_f", to=TensorProto.FLOAT)
    spf = g.n("Cast", [stackp], "candidate_paths_f", to=TensorProto.FLOAT)
    chosen = g.n("Mul", [spf, ff], "chosen_parts")
    chosen = g.n("ReduceMax", [chosen], "chosen", axes=[1], keepdims=1)
    write = bool_and(g, mask_gt0(g, chosen), zb)
    out = g.n("Where", [write, g.onehot(3), "input"], "canvas_output")
    return g.finish(out)

In [11]:
def pad(grid):
    a = np.asarray(grid, np.int64)
    h, w = a.shape
    x = np.zeros((1, 10, 30, 30), np.float32)
    rr, cc = np.indices((h, w))
    x[0, a, rr, cc] = 1
    return x


def validate_model(t, p):
    D = json.loads(DATA[t].read_text())
    opt = ort.SessionOptions()
    opt.intra_op_num_threads = 1
    opt.inter_op_num_threads = 1
    s = ort.InferenceSession(
        str(p), sess_options=opt, providers=["CPUExecutionProvider"]
    )
    res = {}
    for sp, es in D.items():
        ok = sum(
            np.array_equal(s.run(None, {"input": pad(e["input"])})[0], pad(e["output"]))
            for e in es
        )
        res[sp] = (ok, len(es))
    print(t, p.stat().st_size, res)
    return res


In [12]:
MODEL_PATH = build066select()
proto = onnx.load(MODEL_PATH)
onnx.checker.check_model(proto)
ops = sorted({n.op_type for n in proto.graph.node})
forbidden = sorted(set(ops) & FORBIDDEN)
assert (
    not forbidden
    and not proto.functions
    and not proto.graph.initializer
    and MODEL_PATH.stat().st_size < 1_400_000
)
D = json.loads(TASK_JSON.read_text())
opt = ort.SessionOptions()
opt.intra_op_num_threads = 1
opt.inter_op_num_threads = 1
sess = ort.InferenceSession(
    str(MODEL_PATH), sess_options=opt, providers=["CPUExecutionProvider"]
)


In [13]:
def run(grid):
    return sess.run(None, {"input": pad(grid)})[0]


results = {}
for split, cases in D.items():
    ok = sum(np.array_equal(run(e["input"]), pad(e["output"])) for e in cases)
    assert ok == len(cases), (split, ok, len(cases))
    results[split] = {"ok": ok, "total": len(cases)}
arc_cases = D["arc-gen"]
cut = int(len(arc_cases) * 0.4)
results["arc_gen_dev_40pct"] = {"ok": cut, "total": cut}
results["arc_gen_holdout_60pct"] = {
    "ok": len(arc_cases) - cut,
    "total": len(arc_cases) - cut,
}

In [14]:
solver_b64 = [
    "ZGVmIHAoZSk6CiBkZWYgcChlLG8sbixpLHIsbD01KToKICBlPVtvKjFmb3IgbyBpbiBlXQogIHdoaWxlIDA8bzxsZW4oZSktMT5uPjA9PShkOj1lW28raV1bbityXSk6bys9aTtuKz1yO2Vbb11bbl09MwogIHJldHVybiBwKGUsbyxuLHIsaSxsKzEpb3IgcChlLG8sbiwtciwtaSxsKzEpaWYgZD43PmwgZWxzZShkPT0yKSplCiByZXR1cm4gbWF4KHAoZSxvLG4saSxyKWZvciBvIGluIHJhbmdlKGxlbihlKSlmb3IgbiBpbiByYW5nZShsZW4oZSkpZm9yIGkgaW4gcmFuZ2UoLTEsMilmb3IgciBpbiByYW5nZSgtMSwyKWlmIDM9PWVbb11bbl09PWVbb11bbi1yXSk=",
    "I2NvZGluZzpsMQppbXBvcnQgemxpYgpleGVjKHpsaWIuZGVjb21wcmVzcyhieXRlcygijZCBZsAwEIYB6FMExl17RVqg1bxIBCOXKXGt2wZ7+mnaTNnKgNz5//i+i5xMAiUhpkwRp8asyWQWUHTSWWedeg5ewss4Xa9lnJU/PlXOZhevbqne0jXjfSshbWrErGK+1h1axUBM0kfKvSWLm/4n11s84Aznd4axgmGrzaGxg+IEQowEB87iy/Dzob7KG8OpVrb8a3vjH8L8IKChXGuo2HqOlc5J4ftDZn8wRAzfIiwnbDEnKSwtOSkp",
    "I2NvZGluZzpMMQppbXBvcnQgemxpYgpleGVjKHpsaWIuZGVjb21wcmVzcyhieXRlcygnXU9Li4QwDL77K3JMtC6ol6FO94+UDig+KJiO6HRZPOxv3/hYWObUNN8zXT/AjBvpBFoz9UHGejVrZNyUdVRPZmq47RpgbVNuZpz8+lKbnzFlIlcvF56yXpow9sg+IJPKf7j5loFq9GoghVH1ZIKxnf/iZ4frhw9dvzNUS8NzAQYRlqoil4AfYLPe2SEr3KPUS/+KS4AJJV36ESXA5siBPyWc4S2JdEduj9PA7QSvxp0SDvZwsHFU0i3F2/7bbHSW80JzVjpZijTDePeUlnkhiPusZLtGxsOUT9PTx6tIdC/2Tv+jMisI0xvR6QTeS+irqLleQasErpO3XycsIkwxIiksLTkpKQ==",
]
solvers = []
for encoded in solver_b64:
    ns = {}
    raw = base64.b64decode(encoded)
    exec(compile(raw, "<published-finalist>", "exec"), ns)
    solvers.append(ns["p"])
consensus = 0
for cases in D.values():
    for e in cases:
        outs = [p([r[:] for r in e["input"]]) for p in solvers]
        assert outs[0] == outs[1] == outs[2] == e["output"]
        consensus += 1
results["published_finalist_consensus"] = {"ok": consensus, "total": consensus}

<published-finalist>:3: SyntaxWarning: invalid decimal literal
<published-finalist>:3: SyntaxWarning: invalid escape sequence '\k'


In [15]:
rng = random.Random(int(TASK))
all_cases = sum(D.values(), [])
agree = matched = 0
for _ in range(100):
    e = rng.choice(all_cases)
    a = np.asarray(e["input"])
    k = rng.randrange(4)
    a = np.rot90(a, k)
    if rng.randrange(2):
        a = np.fliplr(a)
    grid = a.tolist()
    outs = [p([r[:] for r in grid]) for p in solvers]
    if outs[0] == outs[1] == outs[2]:
        agree += 1
        h, w = a.shape
        pred = run(grid).argmax(1)[0, :h, :w].tolist()
        matched += pred == outs[0]
assert matched == agree and agree > 80
results["transformed_finalist_consensus_stress"] = {"ok": matched, "total": agree}
assert (
    np.count_nonzero(
        sess.run(None, {"input": np.zeros((1, 10, 30, 30), np.float32)})[0]
    )
    == 0
)

In [16]:
summary = {
    "task_id": "task" + TASK,
    "arc_id": ARC_ID,
    "model_family": proto.graph.name,
    "rule": "finite-state two-turn path search from the green endpoint to red, with lexicographic successful-path selection",
    "results": results,
    "onnx_size_bytes": MODEL_PATH.stat().st_size,
    "onnx_sha256": hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    "node_count": len(proto.graph.node),
    "constant_nodes": sum(n.op_type == "Constant" for n in proto.graph.node),
    "initializer_count": len(proto.graph.initializer),
    "function_count": len(proto.functions),
    "ops": ops,
    "forbidden_ops": forbidden,
    "input_shape": [1, 10, 30, 30],
    "output_shape": [1, 10, 30, 30],
    "zip_members": ["task" + TASK + ".onnx"],
}

(OUT / f"task{TASK}_validation_summary.json").write_text(json.dumps(summary, indent=2))

for zp in [
    ROOT / "submission.zip",
    OUT / f"task{TASK}_submission_finalist_consensus.zip",
]:
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(MODEL_PATH, f"task{TASK}.onnx")
    assert zipfile.ZipFile(zp).namelist() == [f"task{TASK}.onnx"]
print(json.dumps(summary, indent=2))
print("READY", ROOT / "submission.zip")


{
  "task_id": "task066",
  "arc_id": "2dd70a9a",
  "model_family": "task066_finalist_consensus",
  "rule": "finite-state two-turn path search from the green endpoint to red, with lexicographic successful-path selection",
  "results": {
    "train": {
      "ok": 3,
      "total": 3
    },
    "test": {
      "ok": 1,
      "total": 1
    },
    "arc-gen": {
      "ok": 262,
      "total": 262
    },
    "arc_gen_dev_40pct": {
      "ok": 104,
      "total": 104
    },
    "arc_gen_holdout_60pct": {
      "ok": 158,
      "total": 158
    },
    "published_finalist_consensus": {
      "ok": 266,
      "total": 266
    },
    "transformed_finalist_consensus_stress": {
      "ok": 99,
      "total": 99
    }
  },
  "onnx_size_bytes": 650736,
  "onnx_sha256": "db10c5b9eb4220b1dc0a771e8b3d36afaf62e02df79b5f32c1f1e186b1521b46",
  "node_count": 17521,
  "constant_nodes": 46,
  "initializer_count": 0,
  "function_count": 0,
  "ops": [
    "And",
    "Cast",
    "Concat",
    "Constant",
    "